In [12]:
from pathlib import Path
import polars as pl
import tensorflow as tf
from jacques import kcqe
import numpy as np
import math
import pandas as pd

import typer
from tqdm import tqdm

import matplotlib.pyplot as plt

from jacques_manuscript.config import MODELS_DIR, PROCESSED_DATA_DIR

In [ ]:
features_path: Path = PROCESSED_DATA_DIR / "train_features.csv"
labels_path: Path = PROCESSED_DATA_DIR / "train_labels.csv"

In [15]:
x_train = pl.read_csv(features_path).to_pandas()

    # Read in labels
y_train = pl.read_csv(labels_path).to_pandas()

In [13]:
tau = tf.constant(np.array([0.1, 0.5, 0.9]), dtype=tf.float32)

kcqe_obj = kcqe.KCQE(x_kernel = "gaussian_diag", p= x_train.shape[1])

block_size = 21

num_blocks = math.floor(y_train.shape[1] / block_size)

In [ ]:
jacques_generator = kcqe_obj.generator(
        x_train_val = x_train,
        y_train_val = y_train,
        batch_size= num_blocks,
        block_size = block_size,
    )

jacques_generator = kcqe_obj.generator(x_train_val = x_train, y_train_val = y_train, batch_size= num_blocks, block_size = block_size,  )

In [17]:
init_param_vec = tf.constant(np.zeros(kcqe_obj.n_param), dtype=np.float32)

In [19]:
param_vec = kcqe_obj.fit(xval_batch_gen = jacques_generator,
            num_blocks = num_blocks,
            tau = tau,
            optim_method = "adam",
            num_epochs = 10,
            learning_rate = 0.1,
            init_param_vec = init_param_vec,
            verbose=True)

In [20]:
param_vec = kcqe_obj.fit(xval_batch_gen = jacques_generator, num_blocks = num_blocks, tau = tau, optim_method = "adam", num_epochs = 10, learning_rate = 0.1, init_param_vec = init_param_vec, verbose=True)